In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def load_model(path, device):
    d = torch.load(path, map_location=device, weights_only=False)
    X = torch.from_numpy(d["X"]).float()
    y = d["y"]
    return X.to(device), y

def load_test(path):
    return pd.read_csv(path).values.astype(np.float32)

def pairwise_distances(x, y):
    x_norm = (x**2).sum(1).view(-1, 1)
    y_norm = (y**2).sum(1).view(1, -1)
    return torch.clamp(x_norm + y_norm - 2.0 * x @ y.t(), 0.0, np.inf)

def infer(train_X, test_X, batch_size, device):
    test_tensor = torch.from_numpy(test_X).float()
    loader = DataLoader(TensorDataset(test_tensor), batch_size=batch_size)
    indices = []
    for (batch,) in loader:
        batch = batch.to(device)
        _, idx = pairwise_distances(train_X, batch).min(0)
        indices.append(idx.cpu())
    return torch.cat(indices).numpy()

def make_submission(sample_path, labels, indices, output_path):
    submission = pd.read_csv(sample_path)
    submission["Label"] = labels[indices]
    submission.to_csv(output_path, index=False)

device = torch.device("cpu")

model_path = "/kaggle/input/models/jek1wantaufik/buddy/pytorch/digit/1/output/knn_store.pt"
test_path = "/kaggle/input/competitions/digit-recognizer/test.csv"
sample_path = "/kaggle/input/competitions/digit-recognizer/sample_submission.csv"
output_path = "/kaggle/working/submission.csv"

train_X, train_y = load_model(model_path, device)
test_X = load_test(test_path)
indices = infer(train_X, test_X, batch_size=512, device=device)
make_submission(sample_path, train_y, indices, output_path)